# 03. Mask Component 지표 계산

synthetic mask를 SegFormer 추론 mask라고 가정하고 connected component별 형상 지표와 contrast metric을 계산한다.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "scratch_postprocess_utils.py").exists():
    matches = list(Path.cwd().glob("**/Scratch_Postprocess/scratch_postprocess_utils.py"))
    if matches:
        NOTEBOOK_DIR = matches[0].parent
sys.path.insert(0, str(NOTEBOOK_DIR))

from scratch_postprocess_utils import *

ensure_dirs()
print("project:", NOTEBOOK_DIR)

In [ ]:
manifest_path = DATA_ROOT / "metadata" / "samples.csv"
if not manifest_path.exists():
    manifest = generate_random_scratch_dataset(n_samples=120, size=640, seed=7, overwrite=True)
else:
    manifest = pd.read_csv(manifest_path)

features = extract_feature_table(manifest, min_area=12)
feature_path = RUNS_ROOT / "component_features.csv"
features.to_csv(feature_path, index=False, encoding="utf-8-sig")
print(feature_path)
display(features.head())

## Contrast metric 요약

In [ ]:
metric_summary = summarize_contrast_metrics(features)
metric_summary.to_csv(RUNS_ROOT / "contrast_metric_summary.csv", index=False, encoding="utf-8-sig")
display(metric_summary)

metric_cols = [
    "luma_contrast_abs",
    "luma_contrast_z",
    "rgb_euclidean_contrast",
    "rgb_contrast_z",
    "max_channel_contrast_abs",
    "mean_channel_contrast_abs",
]
by_color = features.groupby("color_pair")[metric_cols].agg(["min", "median", "max"]).round(4)
by_color.to_csv(RUNS_ROOT / "contrast_metric_summary_by_color_pair.csv", encoding="utf-8-sig")
display(by_color)

## Metric 분포

In [ ]:
metric_cols = [
    "luma_contrast_abs",
    "luma_contrast_z",
    "rgb_euclidean_contrast",
    "rgb_contrast_z",
    "max_channel_contrast_abs",
    "mean_channel_contrast_abs",
]
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, col in zip(axes.ravel(), metric_cols):
    for pair, part in features.groupby("color_pair"):
        ax.hist(part[col], bins=24, alpha=0.55, label=pair)
    ax.set_title(col)
    ax.grid(alpha=0.25)
axes[0, 0].legend(fontsize=8)
plt.tight_layout()
plt.savefig(RUNS_ROOT / "contrast_metric_distributions.png", dpi=150)
plt.show()

## 폭과 contrast 관계

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for pair, part in features.groupby("color_pair"):
    ax.scatter(part["width_px"], part["rgb_euclidean_contrast"], s=18, alpha=0.7, label=pair)
ax.set_xlabel("estimated width_px")
ax.set_ylabel("rgb_euclidean_contrast")
ax.grid(alpha=0.25)
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(RUNS_ROOT / "width_vs_rgb_contrast_scatter.png", dpi=150)
plt.show()